# 02 - Data Cleaning
Validate, correct and enrich the dataset. The same logic lives in `python/data_cleaning.py`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

# Locate the project root whether the notebook is run from notebooks/ or the root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "python"))

import config as cfg
import data_cleaning as dc
import eda_analysis as eda
import visualization as viz

pd.set_option("display.max_columns", 50)
viz.set_theme()

In [ ]:
raw = dc.load_raw()
raw.shape

## 1. Missing values and duplicates

In [ ]:
print('Missing:', int(raw.isna().sum().sum()))
print('Duplicate rows:', int(raw.duplicated().sum()))
print('Duplicate EmployeeNumber:', int(raw['EmployeeNumber'].duplicated().sum()))

## 2. Categorical validation
Compare observed values with the expected domain.

In [ ]:
for col, domain in cfg.CATEGORICAL_DOMAINS.items():
    unexpected = set(raw[col].unique()) - domain
    print(f"{col:<16} unexpected values: {sorted(unexpected) if unexpected else 'none'}")

## 3. Ordinal range validation

In [ ]:
for col, (lo, hi) in cfg.ORDINAL_RANGES.items():
    bad = int(((raw[col] < lo) | (raw[col] > hi)).sum())
    print(f"{col:<26} valid range {lo}-{hi}: out-of-range rows = {bad}")

## 4. Logical consistency
Tenure cannot exceed total working years; the employee must have been old enough to work that long.

In [ ]:
checks = {
    'YearsAtCompany > TotalWorkingYears': (raw.YearsAtCompany > raw.TotalWorkingYears).sum(),
    'YearsInCurrentRole > YearsAtCompany': (raw.YearsInCurrentRole > raw.YearsAtCompany).sum(),
    'YearsSinceLastPromotion > YearsAtCompany': (raw.YearsSinceLastPromotion > raw.YearsAtCompany).sum(),
    'YearsWithCurrManager > YearsAtCompany': (raw.YearsWithCurrManager > raw.YearsAtCompany).sum(),
}
pd.Series(checks, name='violations')

## 5. Outlier check (IQR rule)
Outliers are flagged, **not removed**: very high incomes and long tenures are genuine senior employees.

In [ ]:
cols = ['MonthlyIncome', 'Age', 'YearsAtCompany', 'TotalWorkingYears', 'YearsSinceLastPromotion', 'DistanceFromHome']
display(pd.DataFrame(dc.detect_outliers_iqr(raw, cols)).T)
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, c in zip(axes.ravel(), cols):
    sns.boxplot(x=raw[c], ax=ax, color=cfg.SKY)
    ax.set_title(c)
plt.tight_layout()
plt.show()

## 6. Run the cleaning pipeline

In [ ]:
clean, log = dc.clean(raw)
for step in log['steps']:
    print('-', step)
print('\nShape before:', raw.shape, '| after:', clean.shape)

## 7. Engineered features

In [ ]:
display(clean[['Age', 'AgeGroup', 'MonthlyIncome', 'SalarySlab', 'YearsAtCompany', 'TenureBand',
               'YearsSinceLastPromotion', 'PromotionGap', 'Attrition', 'AttritionFlag']].head(8))

## 8. Save the cleaned dataset

In [ ]:
cfg.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
clean.to_csv(cfg.CLEAN_CSV, index=False)
print('Saved ->', cfg.CLEAN_CSV)